## Multi-Step Workflow using LCEL
Generate business advisor that:
- Accepts industry as input
- Generate business idea
- analyse strength and weakness
- format results as a final report

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field

### 1. instantiate chat model

In [2]:
from dotenv import load_dotenv
load_dotenv('config.env')

llm = ChatOpenAI(
    model = 'gpt-4o-mini',
    temperature=0.0
)

### 2. Creating first Chain
In the end of each chain, we should parse the output and save the logs

In [3]:
logs = []

In [4]:
parser = StrOutputParser()

In [5]:
parse_and_log_output_chain = RunnableParallel(
    output = parser,
    log = RunnableLambda(lambda x: logs.append(x))
)

### 3. idea generation

In [6]:
idea_prompt = PromptTemplate(
    template = (
        "you are a creative business advisor."
        "{industry}"
        "Provide bried decsription of idea."
    )
)

In [7]:
idea_chain = (
    idea_prompt
    | llm
    | parse_and_log_output_chain
)

In [8]:
idea_result = idea_chain.invoke("agriculture")

In [9]:
idea_result["output"]

'**Idea: Vertical Hydroponic Farming for Urban Areas**\n\n**Description:** This innovative business concept focuses on establishing vertical hydroponic farms in urban environments, utilizing unused spaces such as rooftops, warehouses, and vacant lots. The farms will employ advanced hydroponic systems to grow a variety of fresh produce, including leafy greens, herbs, and strawberries, without the need for soil. \n\nBy leveraging technology such as LED grow lights, automated nutrient delivery systems, and IoT sensors for monitoring plant health, the farms can optimize growth conditions and yield high-quality crops year-round. The produce will be sold directly to local restaurants, grocery stores, and through subscription-based delivery services to consumers, promoting farm-to-table freshness.\n\nAdditionally, the business will focus on sustainability by using renewable energy sources, recycling water, and minimizing transportation emissions. Educational workshops and community engagement

In [10]:
logs

[AIMessage(content='**Idea: Vertical Hydroponic Farming for Urban Areas**\n\n**Description:** This innovative business concept focuses on establishing vertical hydroponic farms in urban environments, utilizing unused spaces such as rooftops, warehouses, and vacant lots. The farms will employ advanced hydroponic systems to grow a variety of fresh produce, including leafy greens, herbs, and strawberries, without the need for soil. \n\nBy leveraging technology such as LED grow lights, automated nutrient delivery systems, and IoT sensors for monitoring plant health, the farms can optimize growth conditions and yield high-quality crops year-round. The produce will be sold directly to local restaurants, grocery stores, and through subscription-based delivery services to consumers, promoting farm-to-table freshness.\n\nAdditionally, the business will focus on sustainability by using renewable energy sources, recycling water, and minimizing transportation emissions. Educational workshops and c

In [11]:
for message in logs:
    print(message.content)

**Idea: Vertical Hydroponic Farming for Urban Areas**

**Description:** This innovative business concept focuses on establishing vertical hydroponic farms in urban environments, utilizing unused spaces such as rooftops, warehouses, and vacant lots. The farms will employ advanced hydroponic systems to grow a variety of fresh produce, including leafy greens, herbs, and strawberries, without the need for soil. 

By leveraging technology such as LED grow lights, automated nutrient delivery systems, and IoT sensors for monitoring plant health, the farms can optimize growth conditions and yield high-quality crops year-round. The produce will be sold directly to local restaurants, grocery stores, and through subscription-based delivery services to consumers, promoting farm-to-table freshness.

Additionally, the business will focus on sustainability by using renewable energy sources, recycling water, and minimizing transportation emissions. Educational workshops and community engagement progra

### 4. generated idea analysis

In [12]:
analysis_prompt = PromptTemplate(
    template = (
        "Analyse the follwing business idea:"
        "Idea: {idea}"
        "Identify 3 key stength and weaknesses of the idea."
    )
)

In [13]:
analysis_chain = (
    analysis_prompt
    | llm
    | parse_and_log_output_chain
)

In [14]:
analysis_result = analysis_chain.invoke(idea_result['output'])

In [15]:
analysis_result['output']

'### Strengths\n\n1. **Sustainability and Environmental Impact:**\n   - The vertical hydroponic farming model significantly reduces the carbon footprint associated with traditional agriculture. By utilizing renewable energy sources, recycling water, and minimizing transportation emissions, the business aligns with growing consumer demand for sustainable and environmentally friendly practices. This can enhance brand reputation and attract eco-conscious customers.\n\n2. **Freshness and Quality of Produce:**\n   - By growing produce in urban areas and selling directly to local restaurants and consumers, the business can ensure that the produce is fresher and of higher quality compared to items transported over long distances. This farm-to-table approach can be a strong selling point, appealing to health-conscious consumers and local businesses looking for fresh ingredients.\n\n3. **Utilization of Unused Urban Spaces:**\n   - The ability to transform unused spaces such as rooftops, warehou

In [16]:
logs

[AIMessage(content='**Idea: Vertical Hydroponic Farming for Urban Areas**\n\n**Description:** This innovative business concept focuses on establishing vertical hydroponic farms in urban environments, utilizing unused spaces such as rooftops, warehouses, and vacant lots. The farms will employ advanced hydroponic systems to grow a variety of fresh produce, including leafy greens, herbs, and strawberries, without the need for soil. \n\nBy leveraging technology such as LED grow lights, automated nutrient delivery systems, and IoT sensors for monitoring plant health, the farms can optimize growth conditions and yield high-quality crops year-round. The produce will be sold directly to local restaurants, grocery stores, and through subscription-based delivery services to consumers, promoting farm-to-table freshness.\n\nAdditionally, the business will focus on sustainability by using renewable energy sources, recycling water, and minimizing transportation emissions. Educational workshops and c

In [17]:
for message in logs:
    print(message.content)

**Idea: Vertical Hydroponic Farming for Urban Areas**

**Description:** This innovative business concept focuses on establishing vertical hydroponic farms in urban environments, utilizing unused spaces such as rooftops, warehouses, and vacant lots. The farms will employ advanced hydroponic systems to grow a variety of fresh produce, including leafy greens, herbs, and strawberries, without the need for soil. 

By leveraging technology such as LED grow lights, automated nutrient delivery systems, and IoT sensors for monitoring plant health, the farms can optimize growth conditions and yield high-quality crops year-round. The produce will be sold directly to local restaurants, grocery stores, and through subscription-based delivery services to consumers, promoting farm-to-table freshness.

Additionally, the business will focus on sustainability by using renewable energy sources, recycling water, and minimizing transportation emissions. Educational workshops and community engagement progra

### 5. Report Generation

In [18]:
report_prompt = PromptTemplate(
    template = (
        "Here is the business analysis:"
        "Stength and Weakness: {output}"
        "Generate a structured business report."
    )
)

In [19]:
class AnalysisReport(BaseModel):
    """ Strength and weaknesses about a business idea"""
    strengths: list = Field(default=[], description="Idea's strength list")
    weaknesses: list = Field(default=[], description="Idea's weaknesses list")

In [20]:
report_chain = (
    report_prompt | llm.with_structured_output(AnalysisReport, method="function_calling")
)

In [21]:
report_result = report_chain.invoke(analysis_result['output'])

In [22]:
report_result

AnalysisReport(strengths=['Sustainability and Environmental Impact: The vertical hydroponic farming model significantly reduces the carbon footprint associated with traditional agriculture. By utilizing renewable energy sources, recycling water, and minimizing transportation emissions, the business aligns with growing consumer demand for sustainable and environmentally friendly practices. This can enhance brand reputation and attract eco-conscious customers.', 'Freshness and Quality of Produce: By growing produce in urban areas and selling directly to local restaurants and consumers, the business can ensure that the produce is fresher and of higher quality compared to items transported over long distances. This farm-to-table approach can be a strong selling point, appealing to health-conscious consumers and local businesses looking for fresh ingredients.', 'Utilization of Unused Urban Spaces: The ability to transform unused spaces such as rooftops, warehouses, and vacant lots into prod

### 6. End-to-end chain

In [23]:
e2e_chain = (
    RunnablePassthrough()
    | idea_chain
    | RunnableParallel(idea=RunnablePassthrough())
    | analysis_chain
    | report_chain
)

In [24]:
# %pip install grandalf

In [25]:
import grandalf
graph = e2e_chain.get_graph()
graph.print_ascii()

            +------------------+         
            | PassthroughInput |         
            +------------------+         
                      *                  
                      *                  
                      *                  
              +-------------+            
              | Passthrough |            
              +-------------+            
                      *                  
                      *                  
                      *                  
             +----------------+          
             | PromptTemplate |          
             +----------------+          
                      *                  
                      *                  
                      *                  
               +------------+            
               | ChatOpenAI |            
               +------------+            
                      *                  
                      *                  
                      *           

In [27]:
e2e_result = e2e_chain.invoke("agriculture")

In [28]:
e2e_result

AnalysisReport(strengths=['Sustainability and Resource Efficiency: The vertical hydroponic farming model significantly reduces water usage (up to 90% less than traditional farming) and eliminates the need for soil, making it an environmentally friendly option. This aligns with growing consumer demand for sustainable and eco-friendly food production methods.', 'Local Fresh Produce: By growing food in urban areas and selling directly to local restaurants and consumers, the business can provide fresh produce with a shorter supply chain. This not only enhances the quality and taste of the food but also reduces transportation emissions, contributing to a lower carbon footprint.', 'Community Engagement and Education: Offering workshops and community programs can foster a sense of community and raise awareness about sustainable practices. This can enhance brand loyalty and create a positive public image, attracting customers who value social responsibility.'], weaknesses=['High Initial Invest

In [31]:
e2e_result.strengths

['Sustainability and Resource Efficiency: The vertical hydroponic farming model significantly reduces water usage (up to 90% less than traditional farming) and eliminates the need for soil, making it an environmentally friendly option. This aligns with growing consumer demand for sustainable and eco-friendly food production methods.',
 'Local Fresh Produce: By growing food in urban areas and selling directly to local restaurants and consumers, the business can provide fresh produce with a shorter supply chain. This not only enhances the quality and taste of the food but also reduces transportation emissions, contributing to a lower carbon footprint.',
 'Community Engagement and Education: Offering workshops and community programs can foster a sense of community and raise awareness about sustainable practices. This can enhance brand loyalty and create a positive public image, attracting customers who value social responsibility.']